# Shapley values: one observation and three explanations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/statistics/shapley_values.ipynb)

All data are generated locally. This notebook runs on CPU without credentials or repository files.

Dependencies: numpy, matplotlib, pillow. If needed, install these packages with `%pip install` before running the imports.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

# Keep generated figures in a temporary folder for this notebook session.
figure_folder = TemporaryDirectory()
IMG = Path(figure_folder.name)
COLOURS = ['#167d8d', '#e6a23b', '#526b98']
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False,
                     'axes.spines.right': False, 'figure.dpi': 120})

def show_table(columns, rows):
    header = '| ' + ' | '.join(columns) + ' |'
    separator = '| ' + ' | '.join(['---'] * len(columns)) + ' |'
    body = ['| ' + ' | '.join(map(str, row)) + ' |' for row in rows]
    display(Markdown('\n'.join([header, separator] + body)))

In [ ]:
import math
from matplotlib.animation import FuncAnimation, PillowWriter

## The model and the reference population

The fixed model is f(x)=x0+2x1+x0*x2+0.5*x3². We explain the observation (1,1,2,1), whose prediction is 5.5. This is not a constant model. The zero-mean Gaussian background has unit variances and correlations 0.8 between features 0 and 1 and 0.3 between 0 and 2.

In [ ]:
# Evaluate the fixed quadratic prediction for one or many inputs.
def prediction(x):
    x = np.asarray(x)
    return x[..., 0] + 2 * x[..., 1] + x[..., 0] * x[..., 2] + 0.5 * x[..., 3] ** 2

## Exact coalition expectations

For each subset, interventional (marginal) replacement fixes its observed coordinates and integrates the remaining joint marginal distribution. Conditional replacement instead uses the Gaussian distribution of missing coordinates given those observations.

Inside `coalition_values`, `gain` is Σ_missing,fixed times the inverse of Σ_fixed,fixed. It shifts the missing-feature mean and subtracts the corresponding conditional covariance term. The quadratic expectation adds covariance C02 and half the variance C33, which a plug-in prediction at the mean would miss.

In [ ]:
# Integrate every coalition exactly using Gaussian means and covariance.
def coalition_values(x, mean, cov, conditional=False):
    """Exact Gaussian expectation of the quadratic prediction in every coalition."""
    values = {}
    d = len(x)
    for mask in range(1 << d):
        fixed = np.array([j for j in range(d) if mask >> j & 1], dtype=int)
        missing = np.array([j for j in range(d) if not mask >> j & 1], dtype=int)
        m = mean.copy()
        C = cov.copy()
        if len(fixed):
            if conditional and len(missing):
                gain = cov[np.ix_(missing, fixed)] @ np.linalg.inv(cov[np.ix_(fixed, fixed)])
                m[missing] = mean[missing] + gain @ (x[fixed] - mean[fixed])
                C[np.ix_(missing, missing)] -= gain @ cov[np.ix_(fixed, missing)]
            m[fixed] = x[fixed]
            C[fixed, :] = 0
            C[:, fixed] = 0
        values[mask] = float(prediction(m) + C[0, 2] + 0.5 * C[3, 3])
    return values

In [ ]:
# Weight each marginal contribution by its fraction of feature orders.
def exact_shapley(values, d=4):
    phi = np.zeros(d)
    for j in range(d):
        for mask in range(1 << d):
            if mask >> j & 1:
                continue
            k = mask.bit_count()
            w = math.factorial(k) * math.factorial(d - k - 1) / math.factorial(d)
            phi[j] += w * (values[mask | 1 << j] - values[mask])
    return phi

In [ ]:
# Average incremental contributions across sampled feature orders.
def permutation_shapley(values, n=256, seed=0):
    rng = np.random.default_rng(seed)
    estimates = []
    for _ in range(n):
        row = np.zeros(4)
        mask = 0
        for j in rng.permutation(4):
            nxt = mask | 1 << int(j)
            row[j] = values[nxt] - values[mask]
            mask = nxt
        estimates.append(row)
    return np.cumsum(estimates, axis=0) / np.arange(1, n + 1)[:, None]

In [ ]:
# Compute all three games and assert efficiency, symmetry and the dummy property.
def shapley_experiment():
    x = np.array([1.0, 1.0, 2.0, 1.0])
    mean = np.zeros(4)
    cov = np.eye(4)
    cov[0, 1] = cov[1, 0] = 0.8
    cov[0, 2] = cov[2, 0] = 0.3
    assert np.linalg.eigvalsh(cov).min() > 0
    scenarios = {}
    all_values = {}
    for label, m, conditional in [('Interventional', mean, False), ('Conditional', mean, True), ('Shifted background', np.array([0.5, 0.5, 0.0, 0.0]), False)]:
        vals = coalition_values(x, m, cov, conditional)
        phi = exact_shapley(vals)
        assert abs(phi.sum() - (vals[15] - vals[0])) < 1e-12
        scenarios[label] = {'baseline': vals[0], 'prediction': vals[15], 'phi': phi.tolist()}
        all_values[label] = vals
    test = {mask: sum(([1.0, 1.0, 0.0, 2.0][j] for j in range(4) if mask >> j & 1)) for mask in range(16)}
    assert np.allclose(exact_shapley(test), [1, 1, 0, 2])
    return (scenarios, all_values, permutation_shapley(all_values['Interventional']))

In [ ]:
scenarios, values, estimates = shapley_experiment()
coalition_table = []
for mask in range(16):
    members = [str(j) for j in range(4) if mask >> j & 1]
    label = '{' + ', '.join(members) + '}' if members else 'Empty'
    coalition_table.append([label] + [f'{table[mask]:.6f}'
        for table in values.values()])
show_table(['Coalition', 'Interventional', 'Conditional', 'Shifted background'],
           coalition_table)
allocation_table = []
for name, result in scenarios.items():
    allocation_table.append([name, f"{result['baseline']:.3f}"]
        + [f'{v:.3f}' for v in result['phi']]
        + [f"{result['prediction']:.3f}"])
show_table(['Rule', 'Baseline', 'Feature 0', 'Feature 1', 'Feature 2',
            'Feature 3', 'Prediction'], allocation_table)

## Exact values and a sampled path

The GIF concerns the interventional quadratic game, not the additive implementation test. Amber ticks mark exact values. Sampling error can reach zero and move away again; this path happens to match exactly at 256 orders. Features 0 and 2 share an absolute-error curve, while features 1 and 3 are exact for every order under this rule.

The waterfall adds the 0.8 baseline and the four contributions to recover 5.5. Feature 3 contributes zero because its observed squared term equals its reference expectation.

In [ ]:
# Compare sampled allocations against exact-value ticks, with a final hold.
def shapley_animation(path, truth):
    """A sampled path can hit exact values and move away again."""
    fig, ax = plt.subplots(figsize=(8, 4.5))
    frame_ids = np.unique(np.geomspace(1, len(path), 12).astype(int)) - 1
    frames = list(frame_ids) + [len(path) - 1] * 2

    def update(frame):
        ax.clear()
        ax.bar(range(4), path[frame], color=COLOURS[0])
        for j, exact in enumerate(truth):
            ax.plot([j - 0.32, j + 0.32], [exact, exact], color='#d88a19', lw=3, label='Exact interventional value' if j == 0 else None)
        plural = 'permutation' if frame == 0 else 'permutations'
        ax.set(title=f'Quadratic model: {frame + 1} sampled {plural}', xticks=range(4), xticklabels=[f'Feature {i}' for i in range(4)], ylim=(-0.55, 3.45), ylabel='Estimated contribution')
        ax.legend(loc='upper right', fontsize=9)
        fig.tight_layout()
    return (FuncAnimation(fig, update, frames=frames, interval=1000, repeat=True), fig)

In [ ]:
# Reproduce all article attribution, error and waterfall figures.
def shapley_figures(scenarios, values, path):
    slug = 'shapley-values-from-scratch'
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for i, (label, row) in enumerate(scenarios.items()):
        ax.bar(np.arange(4) + (i - 1) * 0.24, row['phi'], 0.24, label=label, color=COLOURS[i])
    ax.axhline(0, color='black', lw=0.6)
    ax.set(xticks=range(4), xticklabels=['Feature 0', 'Feature 1', 'Feature 2', 'Feature 3'], ylabel='Contribution to prediction')
    ax.text(3, 0.12, 'Zero for this\nobservation', ha='center', fontsize=9)
    ax.set_ylim(-0.1, 2.65)
    ax.legend(fontsize=9)
    fig.tight_layout()
    fig.savefig(IMG / f'{slug}-background.webp')
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(8, 4))
    truth = exact_shapley(values['Interventional'])
    ax.plot(np.arange(1, len(path) + 1), abs(path[:, 0] - truth[0]), label='Features 0 and 2', color=COLOURS[0])
    ax.plot(np.arange(1, len(path) + 1), np.zeros(len(path)), label='Features 1 and 3 (exact from first order)', ls='--', color=COLOURS[1])
    ax.set(xlabel='Sampled permutations', ylabel='Absolute error against exact value')
    ax.legend(fontsize=9)
    fig.tight_layout()
    fig.savefig(IMG / f'{slug}-permutations.webp')
    plt.close(fig)
    fig, ax = plt.subplots(figsize=(8, 4.2))
    row = scenarios['Interventional']
    running = row['baseline']
    ax.bar(0, running, color='#aaa')
    ax.text(0, running + 0.1, f'{running:.2f}', ha='center')
    for j, v in enumerate(row['phi']):
        ax.bar(j + 1, v, bottom=running, color=COLOURS[0])
        running += v
        ax.text(j + 1, running + 0.1, f'{v:+.2f}', ha='center')
    ax.bar(5, row['prediction'], color='#aaa')
    ax.text(5, row['prediction'] + 0.1, f"{row['prediction']:.2f}", ha='center')
    ax.set(xticks=range(6), xticklabels=['Baseline', 'Feature 0', 'Feature 1', 'Feature 2', 'Feature 3', 'Prediction'], ylabel='Model output', ylim=(0, 6.15))
    fig.tight_layout()
    fig.savefig(IMG / f'{slug}-waterfall.webp')
    plt.close(fig)
    animation, fig = shapley_animation(path, truth)
    animation.save(IMG / f'{slug}-convergence.gif', writer=PillowWriter(fps=1))
    plt.close(fig)

In [ ]:
shapley_figures(scenarios, values, estimates)
for name in ['background.webp', 'permutations.webp', 'waterfall.webp', 'convergence.gif']:
    display(Image(filename=str(IMG / f'shapley-values-from-scratch-{name}')))

## Exercises and interpretation

1. Set the off-diagonal covariances to zero and compare conditional with interventional values.
2. Change feature 3 to zero and explain the negative contribution of its squared term.
3. Repeat the permutation estimates with 30 seeds. Compare error at 256 orders rather than relying on the exact endpoint from seed zero.

The game must be specified before its Shapley allocation is calculated. Interventional replacement can describe an intervention on a model's inputs; causal treatment effects in the data-generating world require further assumptions.

References: [Lundberg and Lee](https://arxiv.org/abs/1705.07874), [Aas and colleagues](https://arxiv.org/abs/1903.10464), [Janzing and colleagues](https://proceedings.mlr.press/v108/janzing20a.html).